# Evidencia de la capa Gold: tópico × año

Este notebook documenta la tabla Gold `topic_year` que alimenta el modelo de tópicos emergentes. Es de
solo lectura: lee `data/gold/` (y nada más) y no escribe en `data/`. Conviene montar `data/` en modo de
solo lectura al ejecutarlo. La definición operacional y el diccionario de variables están en
[`documentacion/6.gold.md`](../documentacion/6.gold.md).

El corpus se congeló el 2026-10-09; los valores de este notebook describen ese corte.

## Contenido

1. Manifiesto de Gold.
2. Filas, filas etiquetables y positivos.
3. Distribución de clases por año base.
4. Sensibilidad de la etiqueta: umbral de crecimiento × soporte mínimo.
5. Tamaños y positivos por partición temporal.
6. Variables: tasa de nulos y medias por clase en filas etiquetables.
7. Los 20 tópico-años emergentes con mayor crecimiento, para revisión manual.

In [1]:
from pathlib import Path
import json
import sys

from pyspark.sql import SparkSession, functions as F

GOLD = Path("/data/gold")
PROJECT_ROOT = Path("/app")
if PROJECT_ROOT.exists() and str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.gold_features import FEATURE_COLUMNS
from src.gold_rules import HORIZON_YEARS, emergence_label_column

spark = (
    SparkSession.builder.master("local[*]")
    .appName("gold-evidence")
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

topic_year = spark.read.parquet(str(GOLD / "topic_year")).cache()
labelable = topic_year.filter(F.col("target_emerging").isNotNull()).cache()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/10 13:01:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. Manifiesto

`_manifest.json` se escribe al final de la publicación y describe exactamente las tablas publicadas:
versión de Silver, regla de la etiqueta, universo y ventana de descarga de OpenAlex (instantánea de la
taxonomía).

In [2]:
# 1. The manifest is written last and describes the published tables.
manifest = json.loads((GOLD / "_manifest.json").read_text(encoding="utf-8"))
print(json.dumps({key: value for key, value in manifest.items() if key != "feature_columns"}, indent=2, sort_keys=True))
print(f"feature_columns: {len(manifest['feature_columns'])}")
assert manifest["feature_columns"] == list(FEATURE_COLUMNS)

{
  "created_at": "2026-10-10T12:59:24.838923+00:00",
  "first_year": 2011,
  "gold_version": 2,
  "labeled_rows": 3308,
  "last_year": 2025,
  "positive_rows": 289,
  "rows": 52380,
  "rule": {
    "growth_threshold": 0.5,
    "horizon_years": 2,
    "last_complete_year": 2025,
    "min_support": 10
  },
  "silver_schema_version": 4,
  "source": "openalex",
  "taxonomy_snapshot": {
    "ingested_at_max": "2026-10-09T05:34:07.761256+00:00",
    "ingested_at_min": "2026-10-03T01:26:02.861386+00:00"
  },
  "topic_assignment": "primary_topic",
  "topic_count": 3492,
  "universe_rows": 300171
}
feature_columns: 24


## 2. Filas, etiquetables y positivos

Criterios de aceptación: 52 380 filas (3 492 tópicos × 15 años), sin duplicados de (`topic_id`, `year`),
3 308 filas etiquetables y 289 positivos con los parámetros por defecto.

In [3]:
# 2. Acceptance counts on the published table.
counts = topic_year.agg(
    F.count(F.lit(1)).alias("rows"),
    F.countDistinct("topic_id").alias("topics"),
    F.countDistinct("topic_id", "year").alias("unique_topic_years"),
    F.count("target_emerging").alias("labelable"),
    F.sum("target_emerging").alias("positives"),
).withColumn("positive_rate", F.round(F.col("positives") / F.col("labelable"), 4))
counts.show()
row = counts.first()
assert (row["rows"], row["unique_topic_years"], row["labelable"], row["positives"]) == (52380, 52380, 3308, 289)
bad_end = labelable.filter(F.col("label_end_year") != F.col("year") + HORIZON_YEARS).count()
bad_null = topic_year.filter(((F.col("n_t") < 10) | (F.col("year") > 2023)) & F.col("target_emerging").isNotNull()).count()
print(f"label_end_year != year + {HORIZON_YEARS}: {bad_end}; labeled rows with n_t < 10 or year > 2023: {bad_null}")

+-----+------+------------------+---------+---------+-------------+
| rows|topics|unique_topic_years|labelable|positives|positive_rate|
+-----+------+------------------+---------+---------+-------------+
|52380|  3492|             52380|     3308|      289|       0.0874|
+-----+------+------------------+---------+---------+-------------+



label_end_year != year + 2: 0; labeled rows with n_t < 10 or year > 2023: 0


## 3. Distribución de clases por año base

Los años 2024 y 2025 no tienen etiqueta porque su horizonte termina después de 2025, el último año
completo.

In [4]:
# 3. Class distribution per base year.
(
    topic_year.groupBy("year")
    .agg(
        F.count(F.lit(1)).alias("rows"),
        F.sum(F.when(F.col("n_t") > 0, 1).otherwise(0)).alias("observed_topics"),
        F.count("target_emerging").alias("labelable"),
        F.sum("target_emerging").alias("positives"),
    )
    .withColumn("positive_rate", F.round(F.col("positives") / F.col("labelable"), 4))
    .orderBy("year")
    .show(20)
)

+----+----+---------------+---------+---------+-------------+
|year|rows|observed_topics|labelable|positives|positive_rate|
+----+----+---------------+---------+---------+-------------+
|2011|3492|            976|      113|        3|       0.0265|
|2012|3492|            982|      121|       10|       0.0826|
|2013|3492|           1149|      150|       17|       0.1133|
|2014|3492|           1079|      151|       19|       0.1258|
|2015|3492|           1069|      134|       17|       0.1269|
|2016|3492|           1084|      149|       18|       0.1208|
|2017|3492|           1209|      185|       19|       0.1027|
|2018|3492|           1377|      256|       20|       0.0781|
|2019|3492|           1518|      322|       23|       0.0714|
|2020|3492|           1733|      365|       19|       0.0521|
|2021|3492|           1917|      416|       28|       0.0673|
|2022|3492|           2015|      452|       51|       0.1128|
|2023|3492|           2153|      494|       45|       0.0911|
|2024|34

## 4. Sensibilidad de la etiqueta

Se vuelve a aplicar la misma función Spark de la regla (`emergence_label_column` de
`src/gold_rules.py`) sobre las columnas de prevalencia de `topic_year`, cambiando solo el umbral de
crecimiento y el soporte mínimo. No hay una segunda implementación de la regla. La fila con
`growth = 0.5` y `min_support = 10` debe reproducir 3 308 etiquetables y 289 positivos.

In [5]:
# 4. Sensitivity grid: re-apply the Spark label builder with overridden parameters.
path = [F.col("prevalence_t"), *(F.col(f"label_prevalence_t{step}") for step in range(1, HORIZON_YEARS + 1))]
grid_rows = []
for growth_threshold in (0.30, 0.50, 0.70):
    for min_support in (5, 10, 20):
        label = emergence_label_column(
            F.col("n_t"), path, F.col("year"), growth_threshold=growth_threshold, min_support=min_support
        )
        stats = topic_year.select(label.alias("label")).agg(
            F.count("label").alias("labelable"), F.sum("label").alias("positives")
        ).first()
        grid_rows.append((growth_threshold, min_support, stats["labelable"], int(stats["positives"] or 0)))
sensitivity = spark.createDataFrame(
    grid_rows, "growth_threshold double, min_support int, labelable long, positives long"
).withColumn("positive_rate", F.round(F.col("positives") / F.col("labelable"), 4))
sensitivity.orderBy("growth_threshold", "min_support").show()
default = sensitivity.filter((F.col("growth_threshold") == 0.5) & (F.col("min_support") == 10)).first()
assert (default["labelable"], default["positives"]) == (3308, 289)

+----------------+-----------+---------+---------+-------------+
|growth_threshold|min_support|labelable|positives|positive_rate|
+----------------+-----------+---------+---------+-------------+
|             0.3|          5|     5867|      645|       0.1099|
|             0.3|         10|     3308|      386|       0.1167|
|             0.3|         20|     1746|      196|       0.1123|
|             0.5|          5|     5867|      511|       0.0871|
|             0.5|         10|     3308|      289|       0.0874|
|             0.5|         20|     1746|      145|        0.083|
|             0.7|          5|     5867|      393|        0.067|
|             0.7|         10|     3308|      214|       0.0647|
|             0.7|         20|     1746|      100|       0.0573|
+----------------+-----------+---------+---------+-------------+



**Lectura (corte 2026-10-09).** El soporte mínimo cambia cuántas filas son etiquetables (5 867, 3 308 y
1 746 para 5, 10 y 20), pero casi no mueve la tasa de positivos con un umbral dado. El umbral de
crecimiento sí la mueve: alrededor de 11 % con 0,30, 8,7 % con 0,50 y 6 % con 0,70. Los parámetros por
defecto quedan en el centro de la grilla.

## 5. Particiones temporales

Una fila de entrenamiento es válida para una ventana de evaluación que empieza en el año E cuando
`label_end_year <= E`: su etiqueta ya se conoce al final de E, cuando se hace la predicción.

| Partición | Años base | Regla |
|---|---|---|
| Entrenamiento para validación | 2011–2017 | `label_end_year <= 2019` |
| Validación | 2019–2020 | evaluación |
| Reentrenamiento final | 2011–2020 | `label_end_year <= 2022` |
| Prueba final | 2022–2023 | evaluación |

Los años base 2018 y 2021 no entran en el entrenamiento de su ventana porque su etiqueta termina
después de E.

In [6]:
# 5. Split sizes and positives under the documented rule label_end_year <= E.
splits = [
    ("train_for_validation", F.col("label_end_year") <= 2019),
    ("validation", F.col("year").between(2019, 2020)),
    ("retrain_final", F.col("label_end_year") <= 2022),
    ("test", F.col("year").between(2022, 2023)),
]
split_rows = []
for name, condition in splits:
    part = labelable.filter(condition)
    stats = part.agg(
        F.count(F.lit(1)).alias("rows"), F.sum("target_emerging").alias("positives"),
        F.min("year").alias("first_base_year"), F.max("year").alias("last_base_year"),
    ).first()
    split_rows.append((name, stats["first_base_year"], stats["last_base_year"], stats["rows"], int(stats["positives"])))
(
    spark.createDataFrame(split_rows, "split string, first_base_year int, last_base_year int, rows long, positives long")
    .withColumn("positive_rate", F.round(F.col("positives") / F.col("rows"), 4))
    .show(truncate=False)
)

+--------------------+---------------+--------------+----+---------+-------------+
|split               |first_base_year|last_base_year|rows|positives|positive_rate|
+--------------------+---------------+--------------+----+---------+-------------+
|train_for_validation|2011           |2017          |1003|103      |0.1027       |
|validation          |2019           |2020          |687 |42       |0.0611       |
|retrain_final       |2011           |2020          |1946|165      |0.0848       |
|test                |2022           |2023          |946 |96       |0.1015       |
+--------------------+---------------+--------------+----+---------+-------------+



**Lectura.** La validación tiene una tasa de positivos menor (6,1 %) que su entrenamiento (10,3 %), y la
prueba final una mayor (10,2 %) que su reentrenamiento (8,5 %). Con 42 y 96 positivos, las métricas de
evaluación tendrán intervalos amplios; conviene informar métricas por año base y no solo agregadas.

## 6. Variables en filas etiquetables

Para cada variable numérica de `FEATURE_COLUMNS`: tasa de nulos y media por clase. Los nulos son
intencionales (rezagos antes de 2011, cocientes con denominador cero o nulo); Gold no imputa.
`age_left_censored` se muestra como proporción. `subfield`, `field` y `domain` son categóricas y se
resumen aparte por su número de valores distintos.

In [7]:
# 6. Feature null rates and means by class on labelable rows (no imputation in Gold).
categorical = ("subfield", "field", "domain")
numeric = [name for name in FEATURE_COLUMNS if name not in categorical]
aggregates = []
for name in numeric:
    value = F.col(name).cast("double")
    aggregates += [
        F.avg(F.when(value.isNull(), 1.0).otherwise(0.0)).alias(f"{name}__null"),
        F.avg(F.when(F.col("target_emerging") == 0, value)).alias(f"{name}__mean0"),
        F.avg(F.when(F.col("target_emerging") == 1, value)).alias(f"{name}__mean1"),
    ]
stats = labelable.agg(*aggregates).first()
feature_rows = [
    (name, float(stats[f"{name}__null"]), stats[f"{name}__mean0"], stats[f"{name}__mean1"]) for name in numeric
]
(
    spark.createDataFrame(feature_rows, "feature string, null_rate double, mean_class_0 double, mean_class_1 double")
    .select("feature", *(F.round(name, 4).alias(name) for name in ("null_rate", "mean_class_0", "mean_class_1")))
    .show(len(feature_rows), truncate=False)
)
labelable.agg(*(F.countDistinct(name).alias(f"distinct_{name}") for name in categorical)).show()

+--------------------------+---------+------------+------------+
|feature                   |null_rate|mean_class_0|mean_class_1|
+--------------------------+---------+------------+------------+
|n_t                       |0.0      |43.2849     |40.9758     |
|prevalence_t              |0.0      |0.0029      |0.0026      |
|prevalence_percentile_t   |0.0      |0.9366      |0.9355      |
|prevalence_t_minus_1      |0.0342   |0.0028      |0.0022      |
|prevalence_t_minus_2      |0.0707   |0.0027      |0.002       |
|growth_1y                 |0.0354   |0.2473      |0.2749      |
|growth_2y_past            |0.0789   |0.4017      |0.5497      |
|acceleration              |0.0795   |0.0454      |-0.0155     |
|consecutive_growth_years  |0.0342   |0.9952      |1.2552      |
|years_since_first_observed|0.0      |7.5071      |7.3287      |
|age_left_censored         |0.0      |0.8632      |0.7924      |
|unique_authors_t          |0.0      |149.3647    |125.9619    |
|unique_institutions_t   

+-----------------+--------------+---------------+
|distinct_subfield|distinct_field|distinct_domain|
+-----------------+--------------+---------------+
|              138|            24|              4|
+-----------------+--------------+---------------+



## 7. Tópico-años emergentes para revisión manual

Los 20 positivos con mayor crecimiento de prevalencia en el horizonte (`label_growth_2y`, columna de
auditoría que nunca es variable). Sirve para comprobar que la etiqueta marca temas plausibles y no
artefactos del muestreo.

In [8]:
# 7. Top 20 emergent topic-years by label growth, for manual review.
(
    labelable.filter(F.col("target_emerging") == 1)
    .orderBy(F.desc(f"label_growth_{HORIZON_YEARS}y"), "topic_id", "year")
    .select(
        "topic_name", "subfield", "year", "n_t",
        F.round("prevalence_t", 6).alias("prevalence_t"),
        F.round(f"label_growth_{HORIZON_YEARS}y", 3).alias(f"label_growth_{HORIZON_YEARS}y"),
    )
    .show(20, truncate=60)
)

+---------------------------------------------------+---------------------------------------+----+---+------------+---------------+
|                                         topic_name|                               subfield|year|n_t|prevalence_t|label_growth_2y|
+---------------------------------------------------+---------------------------------------+----+---+------------+---------------+
|               Artificial Intelligence in Education|                    Information Systems|2022|139|    0.004981|          4.475|
|               Advanced Neural Network Applications|Computer Vision and Pattern Recognition|2014| 44|    0.006563|          4.323|
|                Multi-Agent Systems and Negotiation|                Artificial Intelligence|2023| 11|     3.21E-4|          4.004|
|                                     Topic Modeling|                Artificial Intelligence|2022|289|    0.010356|          3.588|
|                      Face recognition and analysis|Computer Vision and Pat

**Revisión manual.** La mayoría de los casos son temas plausibles de crecimiento real en el periodo
(IA en educación, redes neuronales profundas desde 2012–2014, robustez adversarial, IoT y edge
computing). Hay un artefacto visible: «COVID-19 diagnosis using AI» aparece como emergente con años base
2018 y 2019. El tópico es una asignación de OpenAlex hecha con la taxonomía actual, que agrupa
publicaciones previas (por ejemplo, diagnóstico de neumonía por imagen) bajo un nombre posterior. La
etiqueta mide crecimiento de prevalencia del identificador de tópico, no la novedad del nombre; ver las
limitaciones en [`documentacion/6.gold.md`](../documentacion/6.gold.md).

In [9]:
topic_year.unpersist()
labelable.unpersist()
spark.stop()